<!---
Notas de clase Métodos Computacionales
Por 
Óscar Antonio Restrepo Gutiérrez
--->

# Solución de sistemas lineales
 
1. [Operaciones matriciales básicas con python](#Operaciones_matriciales).<br>
2. [Efecto de multiplicar una matriz por un vector](#Efecto_matriz_por_un_vector).<br>
3. [Soluciones sistemas de ecuaciones lineales](#Soluciones_sistemas),<br>
   a. Método matriz inversa.<br>
   b. Método por regla de Cramer.<br>
   c. Eliminación gaussiana.<br>
   d. Métodos iterativos de Jacobi y Gauss-Seidel.<br>
4. [Matriz inversa y Gauss-Jordan](#Inversa_de_una_matriz).<br>
5. [Determinantes](#Determinantes).<br> 
6. [Eficiencia computacional](#comparación_a_rutinas).<br>
7. [Material Complementario](#Complemento).<br>

<a id='Operaciones_matriciales'></a>
## Operaciones matriciales básicas con python

Las rutinas para solución de sistemas lineales y algebra lineal están implementadas como métodos en las librerías `numpy` y `scipy`. Scipy usa librerías optimizadas como ATLAS, LAPACK y BLAS de [algebra lineal](https://docs.scipy.org/doc/scipy/reference/tutorial/linalg.html), Numpy también tiene sus propias librerías,
las librerías se importan como,

```python
import scipy.linalg as LA
import numpy.linalg as La
```

o también,

```python
from numpy import *       
from numpy.linalg import *
from scipy.linalg import * # para algunas funciones como lu, lu_solve, etc 
```

O en notebook use `%pylab` simplemente. 
Para ver la lista de funciones definidas para algebra lineal en scipy ver este [link](https://docs.scipy.org/doc/numpy/reference/routines.linalg.html). Ahora vamos a usarlas:

In [ ]:
%pylab inline
from numpy.linalg import *

#### 1) Dimensión de  matrices 
Un arreglo simple es una matriz de dimensión $n\times1$ o vector, por ejemplo el siguiente array numpy de dimensión 10,
```python
a = array([0,1,2,3,4,5,6,7,8,9])
```
Recordemos que `a == a[:] == a[0:10]`. Para obtener un subarray se usa la siguiente notación,
```python
a[5:8] # da el sub vector array([5, 6, 7])      
```
Por otro lado, las matrices se construyen con arreglos dos dimensiónales (arrays), por ejemplo la siguiente matriz $3\times 3$,
```python
A=array([[0,1,2],[3,4,5],[6,7,8]]) # array([[0, 1, 2],
                                   #        [3, 4, 5],
                                   #        [6, 7, 8]])
```
Para obtener los elementos individuales se usa, `A[i,j]`, donde $i$ barre sobre las filas y $j$ sobre las columnas, así,
```python
print( A[0,0], A[0,1],A[0,2] ) 
print( A[1,0], A[1,1],A[1,2] )
print( A[2,0], A[2,1],A[2,2] )
```
Note que `A[i,:]` da la fila $i$ y `A[:,j]` da la columna $j$. Ejemplo,

```python
A[1,:]   # fila array([3, 4, 5])
A[:,1]   # columna array([1, 4, 7])
A[1]     # igual que A[1,:]
```
Nota: si `A` es un array 2D, puede simplemente escribir `A[i]` para obtener la fila $i$.

#### 2) Submatrices
Con la notación anterior también se pueden obtener submatrices, por ejemplo en la submatriz, 
([1, 2],[4, 5]), $i$ va de $0$ a $2$ en la fila y $j$ de $1$ a $3$ en la columna, ósea,
```python
A[0:2, 1:3] # da array([[1, 2],
            #           [4, 5]])
# También            
A[2, 1:3]   # da array([7, 8])            
```
**Ejercicio**: crear una matriz $5\times 5$, y obtener sus filas y columnas individualmente, y generar submatrices.

In [ ]:
# Hacer tarea
A = np.arange(25).reshape(5,5)
A

#### 3) Permutar filas y columnas
Para permutar filas (o columnas) se hace de la siguiente manera,
```python
A[[0,2]] = A[[2,0]]
```
Permutar columnas 1 and 3
```python
A[:,[0,2]] = A[:,[2,0]]
```
**Ejercicio**: permutar la matriz anterior. 

In [ ]:
# Hacer tarea

#### 4) Operaciones elementales
Sumas y restas se hacen elemento a elemento,
```python
A = arange(9).reshape(3,3) # arange crea array de 9 elementos, reshape lo convierte en matriz 3x3.
B = array([[2,1,3],[-1,2,-1],[3,1,1]])
A+B # da array([[2, 2, 5],
    #           [2, 6, 4],
    #           [9, 8, 9]])
    
A-B # da array([[-2,  0, -1],
    #           [ 4,  2,  6],
    #           [ 3,  6,  7]])
```
En la multiplicación y división numpy falla pues multiplica elemento a elemento $a_{ij}*b_{ij}$ y $a_{ij}/b_{ij}$,
```python
A*B # da array([[ 0,  1,  6], Esto NO es multiplicación matricial
    #           [-3,  8, -5],
    #           [18,  7,  8]])

A/B # da array([[ 0,  1,  0], Esto NO es division matricial o A*inv(B)
    #           [-3,  2, -5],
    #           [ 2,  7,  8]])
```
Numpy también cuenta con la instancia `numpy.matrix()` para poder hacer estas operaciones,
```python
M1 = matrix([[1,2,3],[ 4,5, 6],[7,8,9]]) 
M2 = matrix([[2,1,3],[-1,2,-1],[3,1,1]])

M1 + M2 # da igual que A + B  
M1 - M2 # da igual que A - B
M1 * M2 # da producto matricial de M1 y M2
M1 / M2 # da igual que A/B
```
Note que `A/B` y `M1/M2` no son definidos correctamente (estas operaciones hacen división elemento a elemento), pero se puede computar la inversa de la matriz creada con el array `A` mediante el uso del método `numpy.linalg.inv()`,
```python
inv(A)  # A es matriz singular (no tiene inversa, pues a11 = 0 es cero)

inv(B)  # da array([[-0.17647059, -0.11764706,  0.41176471],
        #           [ 0.11764706,  0.41176471,  0.05882353],
        #           [ 0.41176471, -0.05882353, -0.29411765]])
inv(M2) # da la matriz anterior, igual a inversa de B pero "matrix" en vez de "array".
```
entonces la división matricial es,
```python
M1*inv(M2)     # da matrix([[ 1.29411765,  0.52941176, -0.35294118],
               #            [ 2.35294118,  1.23529412,  0.17647059],
               #            [ 3.41176471,  1.94117647,  0.70588235]])

```
En general, la multiplicación matricial de arrays se puede hacer con varios métodos: `numpy.matmul(), numpy.dot` y el operador `@`,
```python
matmul(A,B)    # si A y B son 2D Arrays
dot(A,B)       # si A y B son 2D Arrays, y producto punto si son 1D arrays
A@B            # igual que matmul o dot. 
```
estos tres métodos son equivalentes. 

Se puede transformar una array a matriz o viceversa,
```python
M1 = np.matrix(A) # dado que A es definido array.
A  = np.array(M1) # dado que M1 es definido Matrix.
```
**Ejercicio**: probar las anteriores operaciones en las matrices `A, B, M1, M2`.

In [ ]:
# hacer tarea

#### 5) Obtener propiedades de una matriz
Para calcular el determinante se usa el método `numpy.linalg.det()` (note que M1 y $A$ son singulares), 
```python
det(A)     # da 0.0, la matriz A es singular y NO tiene inversa.
det(M1)    # da -9.5161973539299405e-16 este valor es muy pequeño ent M1 no tiene 
           # inversa numérica (en algunas máquinas puede dar inv(M1) 
           # pero con números grandes):
           #  matrix([[  3.15251974e+15,  -6.30503948e+15,   3.15251974e+15],
           #          [ -6.30503948e+15,   1.26100790e+16,  -6.30503948e+15],
           #          [  3.15251974e+15,  -6.30503948e+15,   3.15251974e+15]])
det(B) = 17.0
```
Matriz transpuesta de `M1`, note que $a_{ij}$ es cambiado por $a_{ji}$,
```python
transpose(M1) #  método de numpy
```
Obtener diagonal dela matriz,
```python
diagonal(M1)  # da array([1, 5, 9]), método de numpy
```
La diagonal del una matriz,
```python
diag(A)       # da array([1, 5, 9], método de numpy
```
`numpy.diag(array, k)` sirve también para crear una matriz diagonal de dimensión $(n+k)\times(n+k)$, es decir con ceros y una diagonal a la posición `k` a partir de la diagonal principal (si `k` es positivo la diagonal queda en parte superior si no, queda en la parte inferior), veamos, 
```python
# matriz con una diagonal:
diag([3,3],2) # da array([[0, 0, 3, 0],
              #           [0, 0, 0, 3],
              #           [0, 0, 0, 0],
              #           [0, 0, 0, 0]]))
# matriz 5x5 con 3 diagonales:
n = 5        # dimensión de la matriz 
A = diag(arange(1,n+1))+diag(ones(n-2),2)+diag(ones(n-2),-2) 
```
Traza de la matriz o suma de les elementos de la diagonal.
```python
trace(M1)    # da sum(array([1, 5, 9])) = 15

triu(M1)     # Matriz triangular superior
tril(M1)     # Matriz triangular inferior
```
Para obtener la dimensión de la matriz, use,
```python
len(A)       # da 3, numero de filas de la matriz (en un vector da la dimensión)
A.shape      # da (3,3), la dimensión de la matriz.
```
#### 6) Solución de sistemas matriciales 
Para resolver el sistema matricial $A\mathbf{x} = \mathbf{b}$ por cálculo de la matriz inversa: 
```python
b = array([1,1,-1])
x = matmul(inv(B),b) # Multiplicación matricial si B y b tipo array.

b = matrix([1,1,-1]) 
x = inv(M2)*b        # Multiplicación matricial si M2 y b to tipo matrix.
```
También $A\mathbf{x} = \mathbf{b}$  con el método `numpy.solve()`,
```python
x = solve(M2,b)      # Más eficiente que el método anterior.
```
Este último es tal vez el más práctico.
#### 7) Agregar más filas o columnas a una matriz
Para agregar filas a una matriz se usa el comado `numpy.r_[A,B,C,...]` (note que son corchetes cuadrados), o también `numpy.append()` veamos,
```python
b = np.array([5,5,5])# crea una matriz 4x3:
np.r_[A,[b]]         #  array([[0, 1, 2],
                     #         [3, 4, 5],
                     #         [6, 7, 8],
                     #         [5, 5, 5]]) 
np.r_[A,[b,b,b]]     # crea una matriz 6x3
np.r_[A,A]           # crea una matriz 6x3
append(A,b[None,:], axis=0)# agrega fila 
append(A,A, axis=0)  # crea matrix 6x3 (las dos matrices deben tener igual dimensión)
```
Para agregar columnas a una matriz  se usa el método `numpy.c_[A,B,C,...]`, por ejemplo,
```python
b = np.array([5,5,5])# crea una matriz 3x4:
np.c_[A,b]           #  array([[0, 1, 2, 5],
                     #         [3, 4, 5, 5],
                     #         [6, 7, 8, 5]]) 
np.c_[A,b,b,b]       # crea una matriz 3x6
np.c_[b]             # crea vector columna.
append(A,b[:,None], axis=1) # agrega columna
append(A,A, axis=1)  # crea matrix 3x6 (las dos matrices deben tener igual dimensión)
```
También existen los comandos `np.hstack()` y `np.vstack()`, pero es más fácil, recordar los comandos `np.r_[]` y `np.c_[]`.

#### 8) Copias de un array
Note que para hacer una copia de un array o matriz no sirve  hacer `b = a`,
pues `b` solo apunta a la dirección de memoria de `a` (`b` es un alias de `a`) y al 
modificar `b` se modifica `a`, veamos,
```python
b = a                # a es array([0,1,2,3,4,5,6,7,8,9])
b[0] = 5
print(a[0])          # da a[0] = 5
```
Mejor para crear una copia real de `a`, usar el comando `numpy.copy()`,
```python
b = np.copy(a)       # donde a es array([5,1,2,3,4,5,6,7,8,9])
b[0] = 0             # b es array([0,1,2,3,4,5,6,7,8,9])
print(a[0], b[0])    # da a[0] = 5 y b[0] = 0
```
**Ejercicio**: Verificar una a una estas propiedades.

In [ ]:
# Hacer tarea:

In [ ]:
# Tarea: poner las 3 diagonales juntas en esta matriz
#        para que quede como una matriz tridiagonal
n = 5
A = diag(arange(1,n+1))+diag(2*ones(n-2),2)+diag(3*ones(n-2),-2)
A

<a id='Efecto_matriz_por_un_vector'></a>
## Efecto de multiplicar una matriz por un vector
Cuando un vector es multiplicado por un número escalar, el efecto es aumentar o disminuir la magnitud del vector, es decir escala el vector (de ahí la palabra escalar) además si el número es negativo cambia también su sentido. Pero si el vector es multiplicado por una matriz el efecto es $-$además del cambio en su magnitud$-$ un cambio de dirección, por ejemplo en algunos casos significa una rotación (esto también se puede interpretar como un cambio de base según el algebra lineal), para ver esto usemos [arrow](https://matplotlib.org/api/_as_gen/matplotlib.pyplot.annotate.html#matplotlib.pyplot.annotate) para graficar los vectores,
```python
matplotlib.pyplot.arrow(x, y, dx, dy, \*\*kwargs)
```
que grafica un vector de $(x, y)$ a $(x+dx, y+dy)$.

In [ ]:
scalar = -.5
x, y = 4, 2  # vector con coord x,y
arrow(0, 0, x, y, head_width=0.2, head_length=.2, fc='k', ec='k')

#------ vector por escalar --------------------------
v1 = scalar*array([x,y]) # vector escalado a graficar
arrow(0, 0, v1[0], v1[1], head_width=0.2, head_length=.2, fc='r', ec='r')

#------  Matrix por vector --------------------------
#A = np.random.uniform(-1,1,size=(2,2)) # matriz aleatoria 2x2
A = np.random.rand(2,2) # matriz aleatoria 2x2
print(A)
v2 = A@v1               # multiplicar matriz por vector
arrow(0, 0, v2[0], v2[1], head_width=0.2, head_length=.2, fc='b', ec='b')

ylim(-5,5)
xlim(-5,5)
grid()

En general, considere la matriz $A \equiv A(\theta)$, que depende del ángulo $\theta$ y que define una [rotación](https://es.wikipedia.org/wiki/Matriz_de_rotaci%C3%B3n) respecto al eje que pasa por el origen de coordenadas y es perpendicular al plano de rotación (por ejemplo la velocidad angular),

$$
A(\theta) =
\begin{bmatrix}
\cos \theta & -\sin \theta \\[3pt]
\sin \theta & \cos \theta \\
\end{bmatrix},
$$

cualquier vector multiplicado por esta matriz rotará un ángulo $\theta$, veamos:

In [ ]:
def A(theta):
    return np.array([[np.cos(theta),-np.sin(theta)],
                      [np.sin(theta), np.cos(theta)]])

theta = -np.pi/2     
v1 = np.array([3,0])  # vector sin rotar
v2 = A(theta)@v1      # véctor rotado un ángulo theta

figure(1,figsize = (5,5))
arrow(0, 0, v1[0], v1[1], head_width=0.2, head_length=.2, fc='r', ec='r')
arrow(0, 0, v2[0], v2[1], head_width=0.2, head_length=.2, fc='b', ec='b')

ylim(-4,4) # rango en y
xlim(-4,4) # rango en x
grid()

**Tarea**: implementar un reloj donde una flecha sea el minutero y otra el segundero, y que al dar el tiempo (min,seg) este ponga las flechas en el lugar indicado.

**Tarea**: el producto cruz se puede ver como una matriz antisimétrica formada con los elementos del vector $\mathbf{a}$, que al multiplicarse por un vector $\mathbf{b}$ da otro vector que tiene $90^\circ$ grados respecto a los vectores originales, 

$$\mathbf{a} \times \mathbf{b} = [\mathbf{a}]_{\times} \mathbf{b} = \begin{bmatrix}\,0&\!-a_3&\,\,a_2\\ \,\,a_3&0&\!-a_1\\-a_2&\,\,a_1&\,0\end{bmatrix}\begin{bmatrix}b_1\\b_2\\b_3\end{bmatrix}.$$

Mediante esta definición hacer un programa de python que dadas la posición $\mathbf {r}$ y la velocidad $\mathbf {v}$, calcule a) la velocidad angular definida como, 

$$\boldsymbol\omega=\frac{\mathbf r\times\mathbf v}{r^2}=\omega \mathbf {u} ={\frac {d\phi }{dt}}\mathbf {u} ={\frac {v\sin(\theta )}{r}}\mathbf {u},$$

donde $\theta$ es el ángulo entre $\mathbf{r}$ y $\mathbf{v}$ y, $\phi$ y $\mathbf {u}$ son el ángulo y el eje de rotación del cuerpo. b) Calcular la velocidad tangencial del cuerpo dada por,

$$\mathbf{v}_{\perp} =\boldsymbol{\omega} \times\mathbf{r}.$$
<!---
Note que la velocidad angular omega se puede ver como la matrix
$$
\Omega =[\omega ]_{\times }={\begin{bmatrix}\,\,0&\!-\omega _{3}&\,\,\,\omega _{2}\\\,\,\,\omega _{3}&0&\!-\omega _{1}\\\!-\omega _{2}&\,\,\omega _{1}&\,\,0\end{bmatrix}}.
$$
--->

In [ ]:
# Hacer tarea:

<a id='Soluciones_sistemas'></a>
## Solución de sistemas de ecuaciones lineales

Los sistemas lineales son comunes en física, por ejemplo las leyes de Kirchhoff en circuitos generan sistemas lineales de la forma,

$$ a_{11}x_1 + a_{12}x_2 + \cdots a_{1n}x_n = b_1 $$
$$\vdots$$
$$ a_{n1}x_1 + a_{n2}x_2 + \cdots a_{nn}x_n = b_n $$

Existen varias maneras de resolver sistemas lineales de ecuaciones, por inversión matricial, por regla de cramer y por eliminación gaussiana, esta última es de especial interés debido a su fácil implementación computacional, a continuación una breve descripción de cada una de ellas.

### a) Inversión matricial
El primer método es por inversión matricial $A\mathbf{x} = \mathbf{b}$ entonces,

$$A^{-1}A \mathbf{x} = A^{-1}\mathbf{b} $$ 

donde $A^{-1}A = \mathbf{I} \rightarrow  \mathbf{x} = A^{-1}\mathbf{b}$

### b) Regla de Cramer 
para resolver el sistema $A\mathbf{x} = \mathbf{b}$

$$      x_i = \frac{\det A(\mathbf{a}_i\rightarrow \mathbf{b})}{\det A} $$ 

donde $i=b$ indica que se cambia la columna $i$ de la matriz $A$ por vector $b$,
ejemplo en un sistema de 2 ecuaciones:

$$
x_1 = \frac{
\begin{vmatrix}
b_1& a_{12}\\ b_2& a_{22}
\end{vmatrix}
}{
\begin{vmatrix}
a_{11}& a_{12}\\ a_{21}& a_{22}
\end{vmatrix}
}
\quad\hbox{ y }\quad
x_2 = \frac{
\begin{vmatrix}
a_{11}&b_1\\ a_{21}&b_2
\end{vmatrix}
}{
\begin{vmatrix}
a_{11}& a_{12}\\ a_{21}& a_{22}
\end{vmatrix}
}
$$

### c) Eliminación gaussiana por sustitución regresiva

En un sistema de ecuaciones sea $E_i$ con $i=1,2,...,n,$ cada una de las $n$ ecuaciones, entonces
se pueden hacer las siguientes operaciones sobre cada una de las $E_i$
 
  a. multiplicar por un factor $b: bE_i → E_i$ <br>
  b. sumar o restar otra ecuación multiplicada por $b: E_i + bE_j → E_i$ <br>
  c. permutar el orden de dos ecuaciones: $E_i \leftrightarrow E_j$. <br>
 
Comenzamos con un ejemplo,

$$
\begin{array}{r}
E_1 :&  x_1 &+&  x_2 & &      &+& 3x_4 &=& 4,\\
E_2 :& 2x_1 &+&  x_2 &−&  x_3 &+&  x_4 &=& 1,\\
E_3 :& 3x_1 &−&  x_2 &−&  x_3 &+& 2x_4 &=&−3,\\
E_4 :& −x_1 &+& 2x_2 &+& 3x_3 &−&  x_4 &=& 4,\\
\end{array}
$$

si se resta $(E_2 -2E_1) → E_2, (E_3 -3E_1) → E_3, (E_4 + E_1) → E_4$, entonces,

$$
\begin{array}{r}
E_1 :&  x_1&+&  x_2 & &      &+& 3x_4 &=&  4,\\
E_2 :&     &-&  x_2 &-&  x_3 &-& 5x_4 &=&- 7,\\
E_3 :&     &-& 4x_2 &-&  x_3 &-& 7x_4 &=&-15,\\
E_4 :&     &+& 3x_2 &+& 3x_3 &+& 2x_4 &=&  8,\\
\end{array}
$$

luego $(E_3 − 4E_2) → E_3$ y $(E_4 + 3E_2) → E_4$,

$$
\begin{array}{r}
E_1 :&  x_1 &+&  x_2 & &      &+&  3x_4 &=&  4,\\
E_2 :&      &-&  x_2 &-&  x_3 &-&  5x_4 &=& -7,\\
E_3 :&      & &      & & 3x_3 &+& 13x_4 &=& 13,\\        
E_4 :&      & &      & &      &-& 13x_4 &=&-13,\\
\end{array}       
$$

Finalmente vemos que $x_4 = 1$ remplazamos en $E_3$ para encontrar que $x_3 = 0$ y en $E_2$ para dar con $x_2 = 2$ y de $E_1, x_1 = -1$,


En general, considere el sistema de ecuaciones $E_i$,
 
$$
\begin{array}{r} 
E_1 :& a_{11}x_1 &+& a_{12}x_2 &+&\cdots&+& a_{1n}x_n &=& b_1,\\
E_2 :& a_{21}x_1 &+& a_{22}x_2 &+&\cdots&+& a_{2n}x_n &=& b_2,\\
\vdots&          & &           & &\ddots& &           & &      \\
E_n :& a_{n1}x_1 &+& a_{n2}x_2 &+&\cdots&+& a_{nn}x_n &=& b_n,\\
\end{array}
$$

cree la matriz aumentada,
 
$$
\left( \matrix{
a_{11}& a_{12}&...&a_{1n}&\vdots& a_{1,n+1}\\
a_{21}& a_{22}&...&a_{2n}&\vdots& a_{2,n+1}\\
&&\ddots&\\
a_{n1}& a_{n2}&...&a_{nn}&\vdots& a_{n,n+1}\\
}\right) 
$$

donde $a_{i,n+1} = b_i$ para cada $i = 1, 2, · · · , n.$

Si $a_{11}$ distinto de cero, hacemos las operaciones, 
 
$$(E_j − (a_{j1}/a_{11})E_1) → (E_j) \quad\hbox{para cada}\quad j = 2, 3, . . . , n$$

y luego las operaciones (dado que $a_{ii} \neq 0$),

$$(E_j − (a_{ji}/a_{ii})E_i) → (E_j) \quad\hbox{para cada}\quad j = i + 1, i + 2, . . ., n,$$
       
Este procedimiento iterativo da la matriz triangular,

$$
\left( \matrix{
 a_{11}& a_{12} &\cdots& a_{1n}&\vdots& a_{1,n+1}\\
       & a_{22} &\cdots& a_{2n}&\vdots& a_{2,n+1}\\
       &        &\ddots&        &\\
       &        &   & a_{nn}&\vdots& a_{n,n+1}\\
}\right) 
$$

y el nuevo sistema algebraico equivalente tiene la nueva forma triangular,

$$
\begin{array}{r}
        E_1 &:& a_{11}x_1 + a_{12}x_2 + \cdots+a_{1n}x_n &=& a_{1,n+1},\\
        E_2 &:&           + a_{22}x_2 + \cdots+a_{2n}x_n &=& a_{2,n+1},\\
       &       & \ddots \quad\quad\quad&&\\
        E_n &:&                              a_{nn}x_n &=& a_{n,n+1},\\
\end{array} 
$$


claramente $x_n$ es,

$$x_n = a_{n,n+1}/a_{nn}$$

y el siguiente término $x_{n-1}$ es dado por,

$$x_{n−1} = ( a_{n−1,n+1} − a_{n−1,n}x_n )/a_{n−1,n−1}.$$
       
Así en general, las demás $x_i$ se obtienen por sustitución regresiva, ósea,

$$
\begin{eqnarray}
        x_i &=& \frac{a_{i,n+1} − a_{in}x_n − a_{i,n−1}x_{n−1} − · · · −a_{i,i+1}x_{i+1}}{a_{ii}},\\
            &=& \frac{a_{i,n+1} − \sum_{j=i+1}^na_{ij}x_j}{a_{ii}},\\
\end{eqnarray} 
$$



para $i = n − 1, n − 2, · · · , 2, 1$.

El método de eliminación gaussiana es bastante fácil de implementar en computación y es el estándar para solución de sistemas lineales, de hecho este es el método implementado en `linalg.solve(A,b)`. 

**Ejercicio**: Implementar la eliminación gaussiana (Pista: note de las ecuaciones que para construir la matriz triangular hay que hacer dos `for` anidados, uno con $i=1,...,n$ y otro con $j=i+1,...,n$, después de construida la matriz se debe hacer otro `for` con $i=1,...,n$ para obtener los $x_i$. Para el seudocódigo puede ver libro de Burden, página 364). Comparar el resultado al obtenido con la rutina `linalg.solve(A,b)`.

In [ ]:
# Solución: 
# Implementación de rutina de eliminación gaussiana.

def Solve_Gauss(A,b):
    '''
    Resuelve sistema, Ax = b, por eliminación gaussiana.
    '''
    n = len(A)                   # Número de filas de A.
    M = np.c_[A,b]               # Crear matrix aumentada.
    
    #------- Crear matriz triangular: ---------------------------
    for i in range(n):           # Bucle en las filas.
                        
        for j in range(i+1,n):   # Operación Ej - b*Ei -> Ej.
            M[j] = M[j] - (M[j,i]/M[i,i])*M[i] 
                
    #------- Resolver x a partir de la matriz triangular: -------
    x = np.zeros(n)
    x[n-1] = M[n-1,n]/M[n-1,n-1] # Calcular x_n
    for i in range(n-1,-1,-1):   # Calcular x_i con i=n−1,···,2,1
        x[i] = (M[i,n] - sum(M[i,i+1:n]*x[i+1:n]))/M[i,i]

    return M, x   # Retornar matriz triangular y vector x.


A, b = np.random.random((4,4)), np.random.random(4)
D, x = Solve_Gauss(A,b)
D, x, solve(A,b) # Comparación a np.linalg.solve()

### d) Métodos iterativos de Jacobi y Gauss-Seidel
Una cuarta forma de resolver el sistema $A\mathbf{x} = \mathbf{b}$ es por métodos iterados, para esto $A$ se parte como la suma de su matriz diagonal y el resto, ${\mathit {A}}={\mathit {D}}+{\mathit {R}}$, así $({\mathit {D}}+{\mathit {R}})\mathbf{x} = \mathbf{b}$, que da 
$D\mathbf {x}=\left(\mathbf {b} -{\mathit {R}}\mathbf {x}\right)$, lo cual (por método de punto fijo, pues $\mathbf {x}=\mathbf {g}(\mathbf {x})=D^{-1}\left(\mathbf {b} -{\mathit {R}}\mathbf {x}\right)$, pero en forma vectorial) permite definir el método de *Jacobi* por la iteración,

$$
\begin{eqnarray}
\mathbf {x} ^{\left(k+1\right)}&=&{\mathit {D}}^{-1}\left(\mathbf {b} -{\mathit {R}}\mathbf {x} ^{(k)}\right)\\
x_{i}^{\left(k+1\right)}&=&{\frac {1}{a_{ii}}}\left(b_{i}-\sum^n \limits _{j\neq i}{a_{ij}x_{j}^{\left(k\right)}}\right),\quad k=1,2,3,...
\end{eqnarray}
$$

Como aproximación inicial $\mathbf{x}^0$ se puede usar cualquier vector, pero lo ideal es que sea proximo a $\mathbf{x}$, converge si $A$ es diagonal dominante, osea si, $\left|a_{ii}\right|>\sum _{j\neq i}{\left|a_{ij}\right|}$.

El método de *Gauss-Seidel* se define descomponiendo A como $A=L+U$, donde $L$ es la descomposición de la matriz triangular inferior con la diagonal principal de $A$ y $U$ triangular superior con diagonal en ceros, entonces, $A\mathbf{x} = (L+U)\mathbf{x}=\mathbf{b}$ y reorganizando, $L\mathbf{x}=\mathbf{b}- U\mathbf{x}$, que da el procedimiento iterado,

$$
\mathbf {x} ^{(k+1)}=L^{-1}(\mathbf {b}-U\mathbf {x} ^{(k)}).
$$

si se toma ventaja de la forma triangular  $L$, los elementos $x_i^{k+1}$ se pueden calcular secuencialmente entonces la secuencia es,

$$
x_{i}^{(k+1)}={\frac {1}{a_{ii}}}\left(b_{i}-\sum _{j=1}^{i-1}a_{ij}x_{j}^{(k+1)}-\sum _{j=i+1}^{n}a_{ij}x_{j}^{(k)}\right),\quad k=1,2,3,\dots. 
$$
El seudocódigo es,
```julia
function gauss_seidel(A, b, x, kmax)
    for k = 1:kmax
        for j = 1:size(A)
            x[j] = (1/A[j,j])*(b[j] - A[j,:]*x[j] + A[j,j]*x[j])
        end
    end
end
```
Este método es más rápido que jacobi y converge siempre que $A$ se diagonal dominante o que sea simétrica y definida positiva. En general, cualquier método iterado que se escriba de la forma $\mathbf {x}^{(k+1)}=T\mathbf {x}^k+\mathbf {c}$ converge si el autovalor más grande de $T$ es menor que uno, es decir, $|\lambda_{max}|<1.$

**Tarea**: implementar las rutinas de Jacobi y Gauss-Seidel.

In [ ]:
# hacer tarea

<a id='Inversa_de_una_matriz'></a>
# Matrix inversa y Gauss-Jordan
La inversa de una matriz no singular se puede calcular por varios métodos, dos de estos son: por cofactores (es decir mediante determinantes y que es muy costoso computacionalmente como veremos luego) y por eliminación gaussiana, este último se implementa por eliminación de Gauss-Jordan.

## Método de Gauss-Jordan
Considere el sistema $A\mathbf{x} = \mathbf{b}$, el método de Gauss-Jordan consiste en aplicar eliminación gaussiana hasta transformar $A$ en la matriz identidad de tal forma que $\mathbf{b}$ se transforma en $\mathbf{x}$ y sirve para resolver varios sistemas linales de ecuaciones al mismo tiempo, es decir, suponga que se requiere calcular la solución a los siguientes sistemas,

$$
A\mathbf{x}_1 = \mathbf{b}_1,\\ 
A\mathbf{x}_2 = \mathbf{b}_2,\\
\vdots\\
A\mathbf{x}_m = \mathbf{b}_m,
$$

entonces se crea la matriz $B\equiv B(\mathbf{b}_1,\mathbf{b}_2,...,\mathbf{b}_m)$ con columnas $\mathbf{b}_i$ y la matriz $X\equiv X(\mathbf{x}_1,\mathbf{x}_2,...,\mathbf{x}_m)$ con columnas $\mathbf{x}_i$, y que da el sistema $AX=B$ que tiene solución, $X = A^{-1}B$, esto se resuelve por eliminación gaussiana: se crea la matriz aumentada de $A$ y $B$ como, $M \equiv [A\vdots B]_{n,n+m}$, y se aplica eliminación gaussiana hasta que el lado de $A$ de $M$ se transforme en la matriz identidad; este proceso transforma el lado $B$ en $X$. 
Note que si $B = I$ es la matriz identidad entonces $X = A^{-1}$, es decir, este método también sirve para calcular la matriz inversa, veamos, considere el producto $AX = I$, entonces,

$$ AA^{-1} = AX= \left( \matrix{
a_{11} & a_{12} & \cdots & a_{1n} \\
a_{21} & a_{22} & \cdots & a_{2n} \\
       &        & \ddots &        \\
a_{m1} & a_{n2} & \cdots & a_{nn} 
}\right)\left( \matrix{
x_{11} & x_{12} & \cdots & x_{1n} \\
x_{21} & x_{22} & \cdots & x_{2n} \\
       &        & \ddots &        \\
x_{n1} & x_{n2} & \cdots & x_{nn} 
}\right) = 
\left( \matrix{
1 & 0 & \cdots & 0 \\
0 & 1 & \cdots & 0 \\
  &   & \ddots &   \\
0 & 0 & \cdots & 1
}\right)$$


si se usa la definición de la matriz aumentada se puede resolver como,

$$M=\left( \matrix{
a_{11} & a_{12} & \cdots & a_{1n} & \vdots & 1 & 0 & \cdots & 0 \\
a_{21} & a_{22} & \cdots & a_{2n} & \vdots & 0 & 1 & \cdots & 0 \\
       &        & \ddots &        &        &   &   & \ddots &\\
a_{n1} & a_{n2} & \cdots & a_{nn} & \vdots & 0 & 0 & \cdots & 1
}\right),$$

y por eliminación de Gauss-Jordan se convierte la matriz $M$ en $M'$ donde la matriz $A$ se transforma en la matriz identidad, y del lado derecho de $M'$ se obtiene la matriz $X$ con coeficientes $x_{ij}$, 

$$M'=\left( \matrix{
1 & 0 & \cdots & 0 & \vdots & x_{11} & x_{12} & \cdots & x_{1n} \\
0 & 1 & \cdots & 0 & \vdots & x_{21} & x_{22} & \cdots & x_{2n} \\
       &        & \ddots &        &        &   &   & \ddots &\\
0 & 0 & \cdots & 1 & \vdots & x_{n1} & x_{n2} & \cdots & x_{nn}
}\right).$$

### Construcción de la rutina de Gauss-Jordan
Para la construcción de la rutina de Gaus-Jordan, note que en la rutina de eliminación gaussiana se usa la ecuación $E_i$, pare eliminar los términos $a_{ji}$ inferiores de las ecuaciones $E_j$ con $j=i+1, ...,n$, en este caso se debe también usar la misma ecuación $E_i$ para eliminar los términos $a_{ji}$ superiores con $j=1, ...,i-1$ (es decir, en la rutina de eliminación gaussiana anterior haga otro `for j in range(0,i)`); esta técnica se conoce como *Gauss-Jordan*. 

Finalmente, de este cálculo se ve que es más costoso calcular la inversa que resolver directamente el sistema $A\mathbf{x} = \mathbf{b}$ por eliminación gaussiana.

**Ejercicio**: hacer una rutina que calcule la inversa por eliminación gaussiana con Gauss-Jordan al resolver el sistema $AX=B$. 

In [ ]:
# Hacer tarea
def Solve_Gauss_Jordan(A,B):
    '''
    Resuelve sistema, AX = B, por eliminación gaussiana.
    '''
    n = len(A)                   # Número de filas de A.
    M = np.c_[A,B]               # Crear matrix aumentada.
    
    #------- Crear matriz triangular: ---------------------------
    for i in range(n):           # Bucle en las filas.
 
        for j in range(0,i):     # Operación Ej - b*Ei -> Ej.
            M[j] = M[j] - (M[j,i]/M[i,i])*M[i] 

        for j in range(i+1,n):   # Operación Ej - b*Ei -> Ej.
            M[j] = M[j] - (M[j,i]/M[i,i])*M[i] 
                
    #------- Resolver x a partir de la matriz diagonal: -------
    for i in range(0,n):   
          M[i] = M[i]/M[i,i]

    return M[:,n:]    # Retornar matriz triangular y vector x.

n = 4
A, b = np.random.random((n,n)), np.random.random(n)
B = np.eye(n,n)
X = Solve_Gauss_Jordan(A,B)
X, solve(A,B), inv(A) # Comparación a np.linalg: solve() e inv()

Alternativamente, note que lo anterior es equivalente a poner un solo `for` en `j` con `i != j`, es decir no tocamos la diagonal:

In [ ]:
# Alternativa más compacta. 
def Solve_Gauss_Jordan(A,B):
    '''
    Resuelve sistema, AX = B, por eliminación gaussiana.
    '''
    n = len(A)                   # Número de filas de A.
    M = np.c_[A,B]               # Crear matrix aumentada.
    
    for i in range(n):           # Bucle en las filas.
        for j in range(n):       # Operación Ej - b*Ei -> Ej.
            if i != j:
                M[j] = M[j] - (M[j,i]/M[i,i])*M[i] 
                        
    #------- Resolver x a partir de la matriz diagonal: -------
    for i in range(0,n):   
          M[i] = M[i]/M[i,i]

    return M[:,n:]    # Retornar matriz triangular y vector x.

n = 4
A, b = np.random.random((n,n)), np.random.random(n)
B = np.eye(n,n)
X = Solve_Gauss_Jordan(A,B)
X, solve(A,B), inv(A) # Comparación a np.linalg: solve() e inv()

<a id='Determinantes'></a>
# Determinante de una matriz
El determinante es una propiedad escalar de las matrices cuadradas y es útil para calcular la solución de sistemas lineales de ecuaciones, así por ejemplo dicho sistema tiene solución si su determinate es distinto de cero, también es útil para calcular la inversa de una matriz.




## Cálculo de determinantes

Sea $A$ una matriz $n\times n$, con coeficientes $a_{ij}$ entonces:
1. Si $n=1$, $A$ es un escalar $a$ y su determinante es $\det A = a$,
2. Si $n>1$, se define el menor $\det(M_{ij})$ como el determinante de la matriz $(n-1)\times(n-1)$ obtenida de eliminar las fila $i$ y la columna $j$ en $A$.
3. Se define el cofactor $C_{ij}$ asociado al determinante de $M_{ij}$ como $C_{ij} = (-1)^{i+j}\det(M_{ij})$.
4. En general, el determinante of una matriz cuadrada $A$ de $n\times n$ se calcula como,

$$ \det A = \sum_{j=1}^n a_{ij}C_{ij} \quad\hbox{ para todo }\quad i=1,...,n.$$

o también,

$$ \det A = \sum_{i=1}^n a_{ij}C_{ij} \quad\hbox{ para todo }\quad j=1,...,n.$$

dado esto, se calcula la inversa de $A$ como, 

$$A^{-1}= \frac{1}{\det{A}}C^T,$$

donde $C^T$ es la transpuesta de la matriz definida por los cofactores de $C_{ij}$ (que se conoce como la adjunta de  $A$).
Note que el determinante se puede calcular por cualquier fila o columna, lo aconsejable es hacerlo por la que tenga más ceros, como se ve en la fórmula matemática la implementación se puede hacer de manera recursiva en python (pues el determinante depende del menor que es otro determinante) y se fija $i=1$, (se puede escoger cualquier $i$ según la fórmula anterior en 4.), ósea,

$$\det A = \sum_{j=1}^n (-1)^{j}a_{1j} \det(M_{1j})$$

La implementación es (recordar que en python $i$ comienza en 0 y no en 1):

In [ ]:
import numpy as np
from numpy.linalg import det    

# Determinante por cofactores, manera recursiva con i=0
# Muy lenta, proporcional a t ~ n!
def Det(A):
    n = len(A)

    if n==1:
        return A[0,0]
    else:
        d = 0
        sign = -1
        for j in range(n):
            sign = -sign                  # Signo de cofactor
            k = np.delete(range(n),j)     # Remover indece j
            d += sign*A[0,j]*Det(A[1:,k]) # Cofactor
        return d
    
A=np.random.random((5,5))    
Det(A), det(A)            # Comparación a linalg.det()  

### Propiedades de determinantes
De la definición de forma multilineal alternada ([ver complemento](#forma_multilineal_alternada)) se puede demostrar las siguientes propiedades, 

1. Si $A$ tiene todos elementos $a_{ij}=0$,  $\det A = 0$.
2. Si una fila o columna de $A$ es ceros, $\det A = 0$.
3. Si $\hat A$ se obtiene  por la permutación $(E_i)\leftrightarrow (E_j)$, ($i\neq j$) entonces $\det \hat A=-\det A$.
4. Si $\hat A$ se obtiene por el reescalamiento $(\lambda E_i)\leftrightarrow (E_i)$, entonces $\det \hat A=\lambda \det A$.
5. Si $\hat A$ se obtiene por la sustitución $(E_i+\lambda E_j) \leftrightarrow (E_i)$, ($i\neq j$) entonces $\det \hat A=\det A$.
6. $\det(AB)=(\det A)(\det B).$
7. $\det A^t=\det A.$
8. $\det A^{-1}=(\det A)^{-1}$
9. Si $A$ es una matriz triangular superior (o inferior),

$$\det A = \prod_{i=1}^n a_{ii} $$
 
El algoritmo anterior toma un tiempo de $n!$ que es prohibitivo para matrices con $n$ grandes, pero si la matriz es triangular entonces el cálculo es bastante simple y solo requiere $n$ multiplicaciones según la propiedad 9.), por lo tanto se pueden usar las propiedades anteriores con eliminación gaussiana para transformar la matriz $A$ en una triangular y esto solo tiene un costo de $~n^3$ operaciones adicionales.
 
 
**Ejemplo**: en ocho operaciones la matriz $A$ se transforma en la matriz triangular superior $A_8$, es decir,

$$
A=
\left(\begin{array}{r}
2 & 1 & -1 & 1\\
1 & 1 & 0  & 3\\
-1& 2 & 3  &-1\\
3 &−1 & -1 & 2
\end{array}\right)
\quad\Longrightarrow\quad A_8=
\left(\begin{array}{r}
1 &\frac{1}{2}& -\frac{1}{2}& \frac{1}{2}\\
0 &1 & 1 & 5   \\
0 &0 & 3 & 13  \\
0 &0 & 0 & -13
\end{array}\right)
$$

y el determinate de la matriz triangular es -39, se puede calcular como, 

$
\begin{array}{l}
1.& \hbox{La operación } (\frac{1}{2} E_1      \rightarrow     E_1)& \hbox{ da }& \det A_1=\frac{1}{2}\det A\\
2.& \hbox{La operación } (E_2 - E_1            \rightarrow     E_2)& \hbox{ da }& \det A_2= \det A_1=\frac{1}{2}\det A\\
3.& \hbox{La operación } (E_3 + E_1            \rightarrow     E_3)& \hbox{ da }& \det A_3= \det A_2=\frac{1}{2}\det A\\
4.& \hbox{La operación } (E_4 - 3E_1           \rightarrow     E_4)& \hbox{ da }& \det A_4= \det A_3=\frac{1}{2}\det A\\
5.& \hbox{La operación } (2E_2                 \rightarrow     E_2)& \hbox{ da }& \det A_5=2\det A_4=\det A\\
6.& \hbox{La operación } (E_3 - \frac{5}{2}E_2 \rightarrow     E_3)& \hbox{ da }& \det A_6= \det A_5=\det A\\
7.& \hbox{La operación } (E_4 + \frac{5}{2}E_2 \rightarrow     E_4)& \hbox{ da }& \det A_7= \det A_6=\det A\\
8.& \hbox{La operación } (E_3                  \leftrightarrow E_4)& \hbox{ da }& \det A_8=−\det A_7=−\det A\\
\end{array}
$

finalmente el determinante es 39.

**Tarea**: ¿cuántos determinantes hay que calcular para tener la matriz inversa?¿Cuántas operaciones se hacen?

**Tarea**: crear una matriz aleatoria y verificar las propiedades de los determinantes.

**Tarea**: con python muestre que si $A$ es triangular su inversa es triangular y tiene determinante $\det(A^{-1})=1/\det A$.

**Ejercicio**: implementar el cálculo de determinantes mediante eliminación gaussiana y las propiedades anteriores. 


In [ ]:
# Hacer tarea
def Solve_Gauss(A,b):
    '''
    Resuelve sistema, Ax = b, por eliminación gaussiana.
    '''
    n = len(A)                   # Número de filas de A.
    M = np.c_[A,b]               # Crear matrix aumentada.
    
    #------- Crear matriz triangular: ---------------------------
    for i in range(n):           # Bucle en las filas.
                        
        for j in range(i+1,n):   # Operación Ej - b*Ei -> Ej.
            M[j] = M[j] - (M[j,i]/M[i,i])*M[i] 
    
    #------- Determinante matriz triangular: --------------------
    Det=1        
    for i in range(n):
        Det *= M[i,i]
        
    #------- Resolver x a partir de la matriz triangular: -------
    x = np.zeros(n)
    x[n-1] = M[n-1,n]/M[n-1,n-1] # Calcular x_n
    for i in range(n-1,-1,-1):   # Calcular x_i con i=n−1,···,2,1
        x[i] = (M[i,n] - sum(M[i,i+1:n]*x[i+1:n]))/M[i,i]

    return Det, M, x  # Retornar matriz triangular y vector x.


A, b = np.random.random((4,4)), np.random.random(4)
det(A), Solve_Gauss(A,b), solve(A,b) # Comparación a np.linalg.solve()

<a id='comparación_a_rutinas'></a>
# Eficiencia computacional
La eficiencia computacional se refiere a dos cosas, primero al error en los cálculos matriciales y segundo a los tiempos de computación de estas operaciones. Si no hay aproximaciones el error es debido a redondeos de las operaciones aritméticas y el tiempo de computación dependerá del número de estas operaciones: sumas/restas y multiplicaciones/divisiones. Como los tiempos de computación para operaciones de sumas/restas es diferente que para multiplicaciones/divisiones, entonces se deben calcular por aparte (la diferencia dependerá de si los números son enteros (puede ser más rápida la suma) o floats (puede ser más rápida la multiplicación), del compilador usado y de la [arquitectura del procesador](http://nicolas.limare.net/pro/notes/2014/12/12_arit_speed/), por lo que no se puede dar un valor). Veamos primero los efectos del redondeo:

### Errores de redondeo  en la eliminación gaussiana
Aunque el método de eliminación gaussiana es exacto el redondeo de los coeficientes a un determinado número de dígitos significativos tiene un costo computacional y este se manifiesta en las operaciones aritméticas, para entender mejor esto considere el siguiente ejemplo dado en el libro de Burden,

$$
\begin{eqnarray}
E_1 &:& 0.003x_1 &+ 59.14x_2 &= 59.17\\
E_2 &:& 5.291x_1 &- 6.130x_2 &= 46.78 
\end{eqnarray}
$$

Si se considera solo cuatro cifras significativas, 
y se hace la operación $(E_2 + b E_1)\rightarrow (E_2)$ donde 

$$b = -\frac{a_{21}}{a_{11}} = -\frac{5.291}{0.003} = 1763.666\cdots \approx 1764$$

Si se redondea a cuatro cifras (por ejemplo en la segunda ecuación, $-104329.09 \approx -104300$), tenemos,

$$
\begin{eqnarray}
E_1 &:& 0.003x_1 &+ 59.14x_2 &=& 59.17\\
E_2 &:&          &-104300x_2 &=& -104400 
\end{eqnarray}
$$

lo cual da la solución erronea, 

$$
x_1 \approx \frac{59.17 - (59.14)(1.001)}{0.00300} = -10\quad\text{ y }\quad
x_2 \approx 1.001
$$

pero si se usan todos los dígitos,

$$
\begin{eqnarray}
E_1 &:& 0.003x_1& + 59.14x_2            & = & 59.17 \\
E_2 &:& 0       &  -104309.37\bar{6}x_2 & = & -104309.37\bar{6}
\end{eqnarray}
$$

pero la solución exacta es,

$$ x_1 = 10.00 \quad\text{ y }\quad x_2 = 1.00.$$

El error es debido a que $59.14/0.00300 \approx 20000$, lo cual se propaga a travez de las operaciones dando el mal resultado, es decir si $a_{ii}$ es muy pequeño o peor, es cero, el resultado será desastroso. Una manera de minimizar este tipo de errores es hacer permutaciones en las filas (o columnas) para poner el mayor coeficiente $|a_{ji}|$ en el lugar de $|a_{ii}|$ de tal manera que la ecuación que se usa para las operaciones $(E_j − (a_{ji}/a_{ii})E_i → E_j)$, siempre tenga el mayor coeficiente $|a_{ii}|$ de todas las posibles ecuaciones $E_j$, en otras palabras, de las ecuaciones con $j=i,i+1,...,n$, se escoge la ecuación $E_j$ que tenga el mayor coeficiente $|a_{ji}|$ y se permuta con $E_i$ es decir se hace,

$$ |a_{ii}| = \max_{i\leq j\leq n}|a_{ji}|$$

Esta técnica se conoce como *método de pivote*. Por ejemplo en el problema anterior simplemente se permutan las ecuaciones $E_1$ y $E_2$,

$$
\begin{eqnarray}
E_2 &:& 5.291x_1 &- 6.130x_2 &= 46.78,\\ 
E_1 &:& 0.003x_1 &+ 59.14x_2 &= 59.17,\\
\end{eqnarray}
$$

si se repite el problema otras vez con solo cuatro citras significativas da la respuesta correcta, pues ya el divisor es $5.291$. 

Normalmente solo se hace pivote en las filas, si se hace en filas y columnas aumentan las operaciones y el tiempo de computación.

**Ejercicio**: en su rutina de gauss implemente el pivote de filas para reducir el error de redondeo.

In [ ]:
# Hacer tarea

### Eficiencia  en la multiplicación de matrices
En la suma/resta de dos matrices $(A+B)$ de dimension $n\times n$ hay que hacer $n^2$ operaciones (pues cada matriz tiene $n^2$ elementos) el tiempo es obviamente proporcional a $O(n^2)$. No obstante para en la multiplicación matricial $AB$, hay que hacer $n^2$ productos punto (fila $i$ por  columna $j$), puesto que para cada fila y columna hay que hacer $n$ multiplicaciones y $(n-1)$ sumas, entonces el numero de operaciones es dado por $n^2(n+(n-1))=2n^3-n^2 = O(n^3)$; como veremos más adelante algunas técnicas permiten reducir este tiempo hasta el orden de $O(n^2)$.


### Eficiencia para eliminación gaussiana
 Se puede demostrar (Burden página 366) que el número de multiplicaciones y divisiones requeridas en la eliminación gaussiana es,

$$\frac{n^3}{3}+n^2-\frac{n}{3}$$

y que el número de sumas y restas es dado por,

$$\frac{n^3}{3}+\frac{n^2}{2}-\frac{5n}{6}$$

Como se puede ver para valores de $n$ grande el término dominante es $n^3/3$, ósea el tiempo escala como $t\sim O(n^3)$.
En el caso de Gauss-Jordan se puede demostrar que número de multiplicaciones y divisiones es, 
$$\frac{n^3}{2}+n^2-\frac{n}{2}$$

y el número de sumas y restas,

$$\frac{n^3}{2}-\frac{n}{2}$$


**Tarea**: construya matrices de con $n=10,100, 500$ y calcule los tiempos de computación, para el producto y al eliminación gaussiana.

In [ ]:
# hacer tarea

### Tiempos de computación en cálculos matriciales
Usar nuestras propias rutinas de algebra lineal no es lo más eficiente pues la eficiencia depende de la forma cómo se realicen las operaciones aritméticas y del lenguaje de programación, lo mejor es usar las librerías ya implementadas y probadas. Comparemos los tiempos de las librería `scipy.linalg` con las rutinas hechas aquí en python, que se debe hacer de manera estadística, primero miremos como hacer un histograma:

#### Como hacer histogramas
Para generar histograma en el intervalo $[a,b]$ con $M$ bins (barras), considere,

$$\Delta = \frac{b-a}{M}\\ i=\text{int}\left(\frac{x-a}{\Delta}\right)$$

así, cada vez que caiga un valor entre $x$ y $x+\Delta$ se almacena en $h[i]$ como,

$$h[i]=h[i]+1$$

es decir, $h[i]$ cuenta todos los valores que se generan entre $x$ y $x+\Delta$, se puede implementar el código pero el método `plt.hist(x,bins)` hace esto por nosotros (
este método calcula el $\Delta$ como $\frac{x_{max}-x_{min}}{bins}$)

In [ ]:
h=[1,3,4,5,4,3,3,1,3,3,3,1,3,5,4,4,4,1,2,3,6]
# histograma con 6 bins por defecto
hist(h)#, bins=2) # usar 2 o 4 bins

**Ejercicio**: para una matriz aleatoria ($20\times20$) calcular el tiempo promedio de computo para la rutina de gauss implementada en python y la rutina `scipy.solve(A,b)` compare también con `matmul(inv(A),b)` donde `b = np.random.rand(20)`, construya el histograma de los tiempos de cómputo para $500$ cálculos.

In [ ]:
# Calcular tiempo promedio
from datetime import datetime

Nrep = 500 # Numero repeticiones
n = 20     # Dimensión matriz M

def Calcular_Tiempo(n, Nrep):

    Times = np.zeros(Nrep)      # Inicializar array a ceros
    for i in range(Nrep):

        #M = np.array(np.random.random((n,n+1)))
        M = np.array(np.random.random((n,n)))
        b = np.random.random(n)
    
        tstart = datetime.now() # Comenzar tiempo
        # Comparación de tiempos de computo
        # Gaussian_Elimination(M) # Copiar de página de Bustamante
        Solve_Gauss(M,b)  # 1)
        #inv(M)@b          # 2)
        #matmul(inv(M),b)   # 3) 
        #solve(M,b)          # 4) 
 
        tend = datetime.now()  # Terminar tiempo
        
        Times[i] = (tend-tstart).microseconds # Salvar diferencia de tiempos en array
        
    
    print ("Tiempo promedio %lf microsegundos"%(Times.mean()))
    
    #--- Histograma --------
    plt.figure( figsize=(8,5) )
    plt.hist( Times, bins = 30 )
    plt.xlabel( "t μs" )
    plt.ylabel( "Ocurrencias" )
    plt.grid()
    
    return Times.mean()
        
Calcular_Tiempo( n, Nrep )

<a id='Complemento'></a>
# Material Complementario

### Concatenación de matrices: agregar filas y columnas a una matriz
Hay varias maneras de contenar dos matrices o agregar filas y columnas:
`np.append()`, `np.vstack()` y `np.hstack()`, pero la más fácil, usar los comandos `np.r_[A,B,C,...]` y `np.c_[A,B,C,...]` filas y columnas, se usan de la siguiente manera:

In [ ]:
import numpy as np
A=np.array([[1,2,3],[4,5,6],[7,8,9]])
b=np.array([1,1,1])

# Agregar filas 
np.c_[A,b], np.c_[A,b,b,b,A]

In [ ]:
# Agregar columnas (también se puede con np.append())
np.r_[A,[b]], np.r_[A,[b],[b],[b],A]

### Arreglos $n$-dimensionales
Considere la siguiente situación para una matrix $3\times 4$, se desea la suma de cada una de sus filas, pero si se hace `A.sum()` da la suma de todos los $12$ elementos, para hacer esto se hace, useo de `axis`,
```python
A.sum(axis=0) # da array con suma de columnas
A.sum(axis=1) # da array con suma de filas
```
En general para arrays de más de un índice ([areglos $n$-dimensionales](https://docs.scipy.org/doc/numpy/reference/arrays.ndarray.html)),  se debe usar `axis=n` para definir en que dimensión se usa el método, es decir colapsa a a la operación todo los elementos en la dimensión $n$ (ver [link](https://stackoverflow.com/questions/17079279/how-is-axis-indexed-in-numpys-array)),  esto aplica para la gran mayoría de métodos que se definen en numpy: `A.prod()`, `A.mean()`, `A.std()`, `max()`, `min()`,`argmax()`, `argmin()`, etc. Importante, cuando se se use axis por favor verifique que la respuesta es en la dimensión deceada.

**Tarea**: probar con diferentes métodos el uso de axis:

In [ ]:
A = array([[1,1,1,1],
           [2,2,2,4],
           [3,3,3,8]])

A.sum(axis=0), A.sum(axis=1) # suma colunmas y filas
#A.prod(axis=0), A.prod(axis=1)
#A.mean(axis=0), A.mean(axis=1)

### Rutina de eliminación gaussiana con pivote
La siguiente rutina hace eliminación gaussiana con pivote en las filas para reducir el error de redondeo.
Otras rutinas en la página de [Rosettacode](https://rosettacode.org/wiki/Gaussian_elimination#Julia). Note que después del pibote se verifica que `M[i,i]` no sea cero, si esto pasa la matriz es singular.

In [ ]:
def Solve_Gauss(A,b):
    '''
    Resuelve sistema, Ax = b, por eliminación gaussiana.
    '''
    n = len(A)                   # Número de filas de A.
    M = np.c_[A,b]               # Crear matrix aumentada.
    
    #------- Crear matriz triangular: ---------------------------
    for i in range(n):           # Bucle en las filas.
        
        jmax = i                 # Índice del Mji más grande.                                            
        for j in range(i,n):     # Hacer pivote en las filas.          
            if abs(M[j,i]) > abs(M[i,i]): jmax = j 
                
        M[[i,jmax]] = M[[jmax,i]]# Permutar filas i,jmax.

        if M[i,i]==0.0:          # Verificar que la matrix es no singular.
            print("Error, esta matriz es singular.")
            return None, None
                
        for j in range(i+1,n):   # Operación Ej - b*Ei -> Ej.
            M[j] = M[j] - (M[j,i]/M[i,i])*M[i] 
                
    #------- Resolver x a partir de la matriz triangular: -------
    x = np.zeros(n)
    x[n-1] = M[n-1,n]/M[n-1,n-1] # Calcular x_n
    for i in range(n-1,-1,-1):   # Calcular x_i con i=n−1,···,2,1        
        x[i] = (M[i,n] - sum(M[i,i+1:n]*x[i+1:n]))/M[i,i]

    return M, x   # Retornar matriz triangular y vector x.

# M = np.matrix( np.random.random((4,5)))# matriz aleatoria
# D, x = Solve_Gauss(M[:,:-1],M[:,-1])   # falla si M es matrix()
A, b = np.random.random((4,4)), np.random.random(4)
#A = np.arange(16).reshape(4,4)*1.0 # matrix singular
A[0,0] = 0. # falla si no hay pivote.
D, x = Solve_Gauss(A,b)
D, x, solve(A,b) # Comparación a np.linalg.solve()

### Rutina de eliminación por Gauss-Jordan 
Recuerde que el sistema $AA^{-1}=AX=I$ se puede escribir como un sistema de $n$ eciones de la forma,

$$ \left( \matrix{
a_{11} & a_{12} & \cdots & a_{1n} \\
a_{21} & a_{22} & \cdots & a_{2n} \\
\vdots & \vdots & & \vdots\\
a_{n1} & a_{n2} & \cdots & a_{nn} 
}\right)\left( \matrix{
x_{11} \\
x_{21} \\
\vdots \\
x_{n1}
}\right) = 
\left( \matrix{
1 \\
0 \\
\vdots \\
0
}\right),$$

$$\left( \matrix{
a_{11} & a_{12} & \cdots & a_{1n} \\
a_{21} & a_{22} & \cdots & a_{2n} \\
\vdots & \vdots & & \vdots\\
a_{n1} & a_{n2} & \cdots & a_{nn} 
}\right)\left( \matrix{
x_{12} \\
x_{22} \\
\vdots \\
x_{n2}
}\right) = 
\left( \matrix{
0 \\
1 \\
\vdots \\
0
}\right),$$

$$\vdots$$ 
$$\left( \matrix{
a_{11} & a_{12} & \cdots & a_{1n} \\
a_{21} & a_{22} & \cdots & a_{2n} \\
\vdots & \vdots & & \vdots\\
a_{n1} & a_{n2} & \cdots & a_{nn} 
}\right)\left( \matrix{
x_{1n} \\
x_{2n} \\
\vdots \\
x_{nn}
}\right) = 
\left( \matrix{
0 \\
0 \\
\vdots \\
1
}\right).
$$

Este sistema se puede resolver individualmente por eliminación gaussiana, pero la siguiente rutina  usa la definición de la matriz aumentada.

La siguiente rutina hace eliminación gaussiana con el método de Gauss-Jordan (sin pivote, por favor implementarlo).


In [ ]:
def Solve_Gauss_Jordan(A,B):
    '''
    Método de Gauss-Jordan
    Resuelve sistema, AX = B, por eliminación gaussiana 
    donde B es una matriz o un vector, si B = I es la 
    matrix identidad, entonces calcula la inversa de A.
    '''
    n = len(A)                  # Número de filas de A.
    M = np.c_[A,B]              # Crear matrix aumentada.
    
    #------- Crear matriz triangular: ---------------------------
    for i in range(n):          # Bucle en las filas (ecuaciones Ei).
        
        #=========================================
        # Tarea: implementar pivote en filas aquí
        #=========================================
        
        for j in range(0,i):    # Operación Ej - b*Ei -> Ej para poner
            M[j] = M[j] - (M[j,i]/M[i,i])*M[i] # ceros triang superior. 
            
        for j in range(i+1,n):  # Operación Ej - b*Ei -> Ej para poner
            M[j] = M[j] - (M[j,i]/M[i,i])*M[i] # ceros triang inferior. 
       
    for i in range(n):
        M[i,n:] = M[i,n:]/M[i,i]# (lado B) obtener X como M[:,n:] y
        M[i,i] = M[i,i]/M[i,i]  # (Lado A) obtener matriz identidad. 
        # M[i] = M[i]/M[i,i]    # Remplaza lineas previas pero consume 
                                # más tiempo. RECUERDE M[i] es fila i.       

    return M[:,n:]              # retorna X (quita matrix identidad). 


A, b = np.random.random((4,4)), np.random.random(4)
X = Solve_Gauss_Jordan(A,b)
X, solve(A,b) # Comparación a np.linalg.solve()

In [ ]:
# Cálculo de matriz inversa con Gauss-Jordan.
A = np.random.random((4,4))
I = np.eye(4) # matrix identidad

X = Solve_Gauss_Jordan(A,I)
X, inv(A) # Comparación a np.linalg.inv()

In [ ]:
solve(A,eye(4)) # solve(A,B) también sirve

### Comparación del cálculo de tiempo promedio

In [ ]:
# Comparación del cálculo de tiempo promedio de la solución AX=b
# para los métodos: 
#             1) np.solve, 
#             2) inv(M)*b,
#             3) matmul(M,b) 
#             4) Solve_Gauss(M,I)  

# %matplotlib qt
from datetime import datetime

Nrep = 5000 # Numero repeticiones
n = 20      # Dimensión matriz M

def Calcular_Tiempo(n, Nrep):

    Times = np.zeros((4,Nrep))      # Inicializar array a ceros
    for i in range(Nrep):

        #M = np.array(np.random.random((n,n+1)))
        M = np.array(np.random.random((n,n)))
        b = np.random.random(n)
        I = eye(n)              # matriz identidad
        
        tstart = datetime.now() # Comenzar tiempo
        solve(M,b)          # 1) 
        tend = datetime.now()   # Terminar tiempo
        Times[0,i] = (tend-tstart).microseconds # Salvar diferencia de tiempos en array

        tstart = datetime.now() # Comenzar tiempo
        inv(M)@b            # 2)
        tend = datetime.now()   # Terminar tiempo
        Times[1,i] = (tend-tstart).microseconds # Salvar diferencia de tiempos en array
 
        tstart = datetime.now() # Comenzar tiempo
        matmul(inv(M),b)    # 3) 
        tend = datetime.now()   # Terminar tiempo
        Times[2,i] = (tend-tstart).microseconds # Salvar diferencia de tiempos en array
 
        tstart = datetime.now() # Comenzar tiempo
        Solve_Gauss(M,b)   # 4)   muy lenta.
        tend = datetime.now()   # Terminar tiempo
        Times[3,i] = (tend-tstart).microseconds # Salvar diferencia de tiempos en array
       
    meanT = Times.mean(axis=1) # tiempos promedio cada fila
    #--- Histogramas (note que Δ = (x.max-x.min)/bins = (3000)/500 = 6 ) --------
    plt.figure( figsize=(8,5) )
    plt.hist( Times[0], bins = 500, label='%7.3lf μs, np.solve(M,b)'%(meanT[0]),range=(0,3000) )
    plt.hist( Times[1], bins = 500, label='%7.3lf μs, np.inv(M)@b'%(meanT[1])  ,range=(0,3000) )
    plt.hist( Times[2], bins = 500, label='%7.3lf μs, matmul(inv(M),b)'%(meanT[2]) ,range=(0,3000) )
    plt.hist( Times[3], bins = 500, label='%7.3lf μs, Solve_Gauss(M,b)'%(meanT[3]),range=(0,3000) )

    plt.xlabel( "t μs" )
    plt.ylabel( "Ocurrencias" )
    plt.grid()
    plt.legend()

print ("Tiempos promedio microsegundos (μs)")
Calcular_Tiempo( n, Nrep )

In [ ]:
# Comparación del cálculo de tiempo promedio de la inversa de A, 
# mediante la solución AX=B, para los métodos:
#   1) inv(M)
#   2) solve(M,I)
#   3) Solve_Gauss_Jordan(M,I)

from datetime import datetime

Nrep = 5000 # Número repeticiones
n = 20     # Dimensión matriz M

def Calcular_Tiempo(n, Nrep):

    Times = np.zeros((3,Nrep))      # Inicializar array a ceros
    for i in range(Nrep):

        #M = np.array(np.random.random((n,n+1)))
        M = np.array(np.random.random((n,n)))
        b = np.random.random(n)
        I = eye(n)              # matriz identidad
        
        tstart = datetime.now() # Comenzar tiempo
        inv(M)                 
        tend = datetime.now()   # Terminar tiempo
        Times[0,i] = (tend-tstart).microseconds # Salvar diferencia de tiempos en array

        tstart = datetime.now() # Comenzar tiempo
        solve(M,I)         
        tend = datetime.now()   # Terminar tiempo
        Times[1,i] = (tend-tstart).microseconds # Salvar diferencia de tiempos en array
 
        tstart = datetime.now() # Comenzar tiempo
        Solve_Gauss_Jordan(M,I) # muy lenta.
        tend = datetime.now()   # Terminar tiempo
        Times[2,i] = (tend-tstart).microseconds # Salvar diferencia de tiempos en array
    
    meanT = Times.mean(axis=1) # tiempos promedio cada fila
    #--- Histogramas (note que Δ = (x.max-x.min)/bins = (3000)/500 = 6 ) --------
    plt.figure( figsize=(8,5) )
    plt.hist( Times[0], bins = 500, label='%7.3lf μs, inv()     '%(meanT[0]),range=(0,3000) )
    plt.hist( Times[1], bins = 500, label='%7.3lf μs, solve()   '%(meanT[1]),range=(0,3000) )
    plt.hist( Times[2], bins = 500, label='%7.3lf μs, Solve_GJ()'%(meanT[2]),range=(0,3000) )

    plt.xlabel( "t μs" )
    plt.ylabel( "Ocurrencias" )
    plt.grid()
    plt.legend()

print ("Tiempos promedio microsegundos (μs)")
Calcular_Tiempo( n, Nrep )

<a id='forma_multilineal_alternada'></a>
### Nota sobre determinantes

En general se define el determinante como una forma multilineal alternada, es decir, sea $A=f(\mathbf{a_1},...,\mathbf{a_n})$, una matriz función de sus columnas, $\mathbf{a_i}$, entonces se cumple que,

$$
f(\mathbf{a}_1,...,\lambda\mathbf{a}_i+\mathbf{b}_i,...,\mathbf{a}_n)
=\lambda f(\mathbf{a}_1,...,\mathbf{a}_i,...,\mathbf{a}_n)
+f(\mathbf{a}_1,...,\mathbf{b}_i,...,\mathbf{a}_n).\\
f(\mathbf{a}_1,...,\mathbf{a}_i,...,\mathbf{a}_j,...,\mathbf{a}_n)
=-f(\mathbf{a}_1,...,\mathbf{a}_j,...,\mathbf{a}_i,...,\mathbf{a}_n)
$$

La primera propiedad define la linearidad y la segunda la alternación con respecto al signo.

En términos geométricos el determinante define el volumen orientado del espacio $n$ dimensional, así por ejemplo para el espacio $3D$, el valor absoluto del determinante da el volumen de paralelepípedo definido por los tres vectores, $\mathbf{a},\mathbf{b},\mathbf{c}$, así,

$$
\det A(\mathbf{a},\mathbf{b},\mathbf{c})=\begin{vmatrix} a_1 & b_1 & c_1\\ a_2 & b_2 & c_2\\ a_3 & b_3 & c_3
\end{vmatrix}.
$$

Note que si $A$ es una matriz con coeficientes generados con distribución uniforme entonces cuando $n\rightarrow \infty, \det(A)\rightarrow 0$.
Algunas rutinas adicionales para determinantes:

In [ ]:
# Manera rápida de calcular el determinante (pero más rápido 
# linalg.det(A)) por eliminación gaussiana con pivote en filas.

def Det_gauss(A):
    '''
    Calcula determinante por eliminación gaussiana.
    '''
    n = len(A)                   # Número de filas de A.
    M = np.copy(A)               # Crear matrix aumentada.
    
    #------- Crear matriz triangular: ---------------------------
    sign = 1.
    for i in range(n):           # Bucle en las filas.
        
        jmax = i                 # Índice del Mji más grande, para
        for j in range(i+1,n):   # hacer pivote en las filas.          
            if abs(M[j,i]) > abs(M[i,i]): 
                jmax = j 

        M[[i,jmax]] = M[[jmax,i]]# Permutar filas i,jmax.
        if jmax != i: sign = -sign # Cambiar signo si hay permutación.
        if M[i,i] == 0.0: return 0.0 # mirar si la matriz es singular.
            
        for j in range(i+1,n):   # Operación Ej - b*Ei -> Ej.
            M[j] = M[j] - (M[j,i]/M[i,i])*M[i] 
 
    Det = 1.                                         
    for i in range(n): 
        Det *= M[i,i]            # determinante de matrix triag

    return sign*Det

A = np.random.random((100,100))
A = np.arange(16).reshape(4,4)*1.0# matriz singular, A debe ser float.
Det_gauss(A), det(A)             # comparación a linalg.det() 

In [ ]:
%timeit Det_gauss(A)

In [ ]:
# Manera rápida de calcular el determinante (pero más rápido 
# linalg.det(A)) por eliminación gaussiana con pivote en filas.
# Mejor alternativa que la anterior debido a numba.
from numba import njit

@njit
def Det_gauss(A):
    '''
    Calcula determinante por eliminación gaussiana con pivote de filas.
    '''
    n = len(A)                   # Número de filas de A.
    M = np.copy(A)               # Crear matrix aumentada.
    
    #------- Crear matriz triangular: ---------------------------
    sign = 1.
    for i in range(n):           # Bucle en las filas.
        
        jmax = i                 # Índice del Mji más grande, para
        for j in range(i+1,n):   # hacer pivote en las filas.          
            if abs(M[j,i]) > abs(M[i,i]): 
                jmax = j 

#        M[[i,jmax]] = M[[jmax,i]]# Permutar, no funciona con numba.
        swap = np.copy(M[i,:])   # Permutar filas i con imax 
        M[i,:] = M[jmax,:]
        M[jmax,:] = swap

        if jmax !=i: sign = -sign# Cambiar signo si hay permutación.
        if M[i,i] == 0.0: return 0.0 # mirar si la matriz es singular.       
        
        for j in range(i+1,n):   # Operación Ej - b*Ei -> Ej.
            M[j] = M[j] - (M[j,i]/M[i,i])*M[i] 
 
    Det = 1.                                         
    for i in range(n): 
        Det *= M[i,i]            # determinante de matrix triag

    return sign*Det

A = np.random.random((100,100))
Det_gauss(A), det(A)             # comparación a linalg.det() 

In [ ]:
# Manera rápida de calcular el determinante por eliminación gaussiana.
# La más rápida si se usa numba, pero menos precisa pues pivote solo si M[i,i]=0.
# https://integratedmlai.com/find-the-determinant-of-a-matrix-with-pure-python-without-numpy-or-scipy/

from numba import njit
@njit
def Det_fast(A):
    # Section 1: Establish n parameter and copy A
    n = len(A)
    M = np.copy(A)
    sign = 1. 
    # Section 2: Row ops on A to get in upper triangle form
    for i in range(n):           # A) i stands for focus diagonal
        for j in range(i+1,n):   # B) only use rows below i row
            if M[i,i] == 0:      # C) if diagonal is zero ...
#                M[i,i] = 1.0e-10 # change to ~zero( but NOT GOOD IDEA IF n IS LARGE), then:

                # swap row i with row jmax (this is pivoting technique):
                jmax = np.argmax(np.abs(M[i:,i])) + i # get the row index of the largest elem

                swap = np.copy(M[i,:]) # do permutation of row i with row jmax: 
                M[i,:] = M[jmax,:]
                M[jmax,:] = swap
            
                sign = -sign     # change sign after permutation
            
            Scal = M[j,i]/M[i,i] # D) cr stands for "current row"            
            for k in range(n):   # E) cr - Scal * iRow, one element at a time
                M[j,k] = M[j,k] - Scal*M[i,k]
     
    # Section 3: Once M is in upper triangle form ...
    prod = 1.0
    for i in range(n):        
        prod *= M[i,i]           # product of diagonals is determinant

    return sign*prod


A = np.random.random((100,100))
A[0,0]= 0.         # case 1, not good when M[i,i] = 1.0e-10 and worse if M[i,i] = 1.0e-18
#A[0,0] =  1.0e-15 # case 2, the two functions differs
#A=np.array([[0,1.],[1.,4.]])#

#A=np.array([[0.003,59.14],[5.291,-6.130]])# example of pivoting
Det_fast(A), det(A)# Comparación a linalg.det()

In [ ]:
# algoritmo 2) (muy lenta, es por cofactores)
# from 
# https://stackoverflow.com/questions/47465356/how-to-find-determinant-of-matrix-using-python
# Warning: In Python accessing a "nested list" cannot be done by multi-dimensional slicing, 
# i.e.: A[0,0], instead one would write A[0][0], this is because m is a list and not an array.

import numpy as np

def determinant(A, mul=1):
 d = len(A)
 if d == 1:
    return mul*A[0][0] 
 else:
    sign = -1
    sum = 0
    for i in range(d):
        m = []
        for j in range(1, d):
            buff = [] 
            for k in range(d):
                if k != i:
                    buff.append(A[j][k])
            m.append(buff)
        sign *= -1
        sum += mul*determinant(m, sign*A[0][i])# here we introduce a list, not an array
    return sum

A = np.array([[1,-2,3],[0,-3,-4],[0,0,-3]])
#A=np.random.random((10,10))# takes long time! 
determinant(A)

In [ ]:
# Comparación de tiempos para las tres rutinas cálculo de determinantes

%timeit Det_fast(A) # muy rápida con numba (pivote solo si Mii=0), la más lenta sin numba
%timeit Det_gauss(A)# 3 veces más lenta devido a pivote 
%timeit det(A)      

In [ ]:
# Calcular elemento más común en un array
a=np.array([0,0,0,0,1,1,2,2,2,2,2,2,2,1,1,1,1,1])
np.argmax(np.bincount(a))
#scipy.stats.mode(a) # this is slower
#np.bincount(a)

In [ ]:
# Método de jacobi, para garantizar convergencia, A, debe 
# ser diagonal dominante: |a_ii| > ∑_j≠i |a_ij|

import numpy as np

def jacobi(A, b, x0, eps=1e-10, maxiter=500): 
    D = np.diag(np.diag(A))  # Matriz con diagonal de A.
    R = A - D                # Restar diagonal.
    xold = x0 
    for i in range(maxiter): 
        Dinv = np.diag(1./np.diag(D)) # Inversa de D.
        xnew = np.dot(Dinv, b-np.dot(R, xold)) 
        dx = xnew-xold
        if np.dot(dx, dx) < eps**2.: break
        xold = xnew 
    return xnew 
 
# A debe ser diagonal dominante (elinar diag y ver que pasa):
A = np.random.rand(4,4) + np.diag([2,2,2,2])
b = np.array([9, 3, 5, 1]) 
 
x0 = np.zeros(4) 
x = jacobi(A, b, x0) 
 
x, np.linalg.solve(A,b) 

### Nota sobre aritmética computacional 
Is multiplication slower or faster than addition on modern CPUs?

*In integer arithmetic*, addition is usually appreciably faster. It has been observed differences of the order of 3 times faster, more for 8-byte objects. 

*In real arithmetic*, multiplication may be faster for the following reason:
When two real numbers are multiplied, the mantissae are multiplied together and the exponents are added, and these operations can be carried out in parallel. When two real numbers are added, first the mantissa of the smaller number must be shifted so that the exponents match (a process termed normalisation). Then the mantissae must be added. The result of the addition may overflow the original word length by 1 bit, or it may generate any number of leading zeros. Therefore the result must be normalised again. There are therefore 3 steps and they must be done in series.